In [1]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import platform
from pprint import pprint
import pandas as pd                # 데이터프레임 처리
import numpy as np                 # 수치 계산
import re
import matplotlib.pyplot as plt    # 시각화
import seaborn as sns              # 고급 시각화 (혼동행렬 히트맵 등)
from tqdm import tqdm              # 진행률 표시 바
import torch                       # PyTorch (딥러닝 프레임워크)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score  # 평가 지표
from transformers import pipeline  # HuggingFace 파이프라인 (모델을 쉽게 사용)
from torchinfo import summary  # 모델 구조 요약 (파라미터 수, 레이어별 크기)
import warnings
warnings.filterwarnings('ignore')  # 불필요한 경고 메시지 숨기기

# ========================================
# 운영체제별 한글 폰트 설정
# ========================================
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

/Users/jin/Develop/codingclub/game-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


라이브러리 로드 완료


In [2]:
# 설치된 라이브러리 버전과 실행 환경을 확인합니다
import transformers

print(f"transformers 버전: {transformers.__version__}")
print(f"torch 버전: {torch.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  GPU 이름: {torch.cuda.get_device_name(0)}")
else:
    print("  => CPU 모드로 실행됩니다 (GPU보다 느리지만 실습에는 충분합니다)")

# 참고: 현재 torch는 CPU 전용 버전(torch+cpu)으로 설치되어 있습니다.
# GPU 버전 torch를 설치해도 CPU에서 동일하게 실행 가능하지만,
# CPU 전용 버전은 용량이 훨씬 작아(~150MB vs ~2.5GB) 현재 실습 환경에서 해당 버전을 이용했습니다.

transformers 버전: 5.7.0
torch 버전: 2.11.0
GPU 사용 가능: False
  => CPU 모드로 실행됩니다 (GPU보다 느리지만 실습에는 충분합니다)


In [ ]:
# model= 을 지정하지 않으면 해당 과제의 기본 모델이 자동 선택됩니다 (출력 경고 메시지에서 모델을 확인할 수 있음)
en_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512,
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 32556.68it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[모델 출력 원본]
[{'label': 'positive', 'score': 0.984351634979248}]

[파이프라인 출력 형태]
  타입: <class 'dict'>
  내용: {'label': 'positive', 'score': 0.984351634979248}
  라벨(label): positive
  확신도(score): 0.9844

* label: POSITIVE(긍정) 또는 NEGATIVE(부정)
* score: 해당 라벨에 대한 모델의 확신도 (1에 가까울수록 확신)


In [4]:
df_reviews = pd.read_csv('../../../../data/preprocessed/steam_indie_reviews.csv')
print('데이터 로드 완료.')
print(f"리뷰: {df_reviews.shape}")
df_reviews.info()

데이터 로드 완료.
리뷰: (166875, 24)
<class 'pandas.DataFrame'>
RangeIndex: 166875 entries, 0 to 166874
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   recommendationid               166875 non-null  int64  
 1   appid                          166875 non-null  int64  
 2   language                       166875 non-null  str    
 3   review                         166875 non-null  str    
 4   timestamp_created              166875 non-null  int64  
 5   timestamp_updated              166875 non-null  int64  
 6   voted_up                       166875 non-null  bool   
 7   votes_up                       166875 non-null  int64  
 8   votes_funny                    166875 non-null  int64  
 9   weighted_vote_score            166875 non-null  float64
 10  comment_count                  166875 non-null  int64  
 11  steam_purchase                 166875 non-null  bool   
 12  received_for_

In [5]:
df_sample = df_reviews[(df_reviews['appid'] == 324470) & (df_reviews['language'] == 'english')]
print(df_sample.shape)
df_sample.head(3)

(92, 24)


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,author_steamid,author_num_games_owned,author_num_reviews,author_last_played,created_date,updated_date,author_last_played_date,playtime_forever_hours,playtime_last_two_weeks_hours,playtime_at_review_hours
1,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",1445885616,1445885616,True,1,0,0.421372,...,76561198049920411,0,3,1445958836,2015-10-26 18:53:36,2015-10-26 18:53:36,2015-10-27 15:13:56,0.216667,0.0,0.216667
2,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,0.500076,...,76561198047893068,435,17,1623275518,2015-10-26 19:03:37,2016-12-24 01:02:18,2021-06-09 21:51:58,13.616667,0.0,12.666667
3,18700348,324470,english,Ever played Bhop? Surf? If so this games mecha...,1445889159,1445895001,True,16,0,0.637511,...,76561198045694190,0,3,1541537929,2015-10-26 19:52:39,2015-10-26 21:30:01,2018-11-06 20:58:49,7.483333,0.0,0.916667


In [6]:
df_test = df_reviews[(df_reviews['appid'] == 324470) & (df_reviews['language'] == 'english')]

# 데이터 정리: 결측치, 빈 문자열, 10자 미만 리뷰를 제거합니다
df_test = df_test.dropna(subset=['review'])
df_test = df_test[df_test['review'].str.strip() != '']
df_test = df_test[df_test['review'].str.len() >= 10].reset_index(drop=True)

print(f"[전체 테스트 데이터] {len(df_test):,}건")
print(f"  긍정: {df_test['voted_up'].sum():,}건")
print(f"  부정: {(~df_test['voted_up']).sum():,}건")

df_sample = df_test.reset_index(drop=True)

# BERT용 전처리: BBCode 태그 제거 + 반복 문자 축소
def clean_for_bert(text: str) -> str:
    text = re.sub(r'\[.*?\]', '', text)          # BBCode 태그 제거 ([b], [/b], [h1] 등)
    text = re.sub(r'(.)\1{4,}', r'\1\1', text)  # 5회 이상 반복 문자 → 2회로 축소
    return text.strip()

df_sample['review_clean'] = df_sample['review'].apply(clean_for_bert)

print(f"\n[샘플링 완료] {len(df_sample)}건")
print(f"  긍정: {df_sample['voted_up'].sum():,}건")
print(f"  부정: {(~df_sample['voted_up']).sum():,}건")
print(f"\n전처리 예시:")
print(f"  원본: {df_sample['review'].iloc[5]!r}")
print(f"  정제: {df_sample['review_clean'].iloc[5]!r}")
display(df_sample.head())

[전체 테스트 데이터] 87건
  긍정: 67건
  부정: 20건

[샘플링 완료] 87건
  긍정: 67건
  부정: 20건

전처리 예시:
  원본: 'Enjoying this game [b]SO MUCH[/b] !'
  정제: 'Enjoying this game SO MUCH !'


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,author_num_games_owned,author_num_reviews,author_last_played,created_date,updated_date,author_last_played_date,playtime_forever_hours,playtime_last_two_weeks_hours,playtime_at_review_hours,review_clean
0,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",1445885616,1445885616,True,1,0,0.421372,...,0,3,1445958836,2015-10-26 18:53:36,2015-10-26 18:53:36,2015-10-27 15:13:56,0.216667,0.0,0.216667,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe..."
1,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,0.500076,...,435,17,1623275518,2015-10-26 19:03:37,2016-12-24 01:02:18,2021-06-09 21:51:58,13.616667,0.0,12.666667,"this game is like a zen-garden, I love it! \n\..."
2,18700348,324470,english,Ever played Bhop? Surf? If so this games mecha...,1445889159,1445895001,True,16,0,0.637511,...,0,3,1541537929,2015-10-26 19:52:39,2015-10-26 21:30:01,2018-11-06 20:58:49,7.483333,0.0,0.916667,Ever played Bhop? Surf? If so this games mecha...
3,18702904,324470,english,"Greatness comes in all sorts of things, but th...",1445899986,1445903284,True,3,1,0.469881,...,1241,1122,1447904671,2015-10-26 22:53:06,2015-10-26 23:48:04,2015-11-19 03:44:31,0.833333,0.0,0.650000,"Greatness comes in all sorts of things, but th..."
4,18707357,324470,english,Don't buy this game. The platforms you jump on...,1445929789,1445999928,False,8,0,0.300778,...,450,20,1445999844,2015-10-27 07:09:49,2015-10-28 02:38:48,2015-10-28 02:37:24,0.200000,0.0,0.200000,Don't buy this game. The platforms you jump on...


In [7]:
texts = df_sample['review_clean'].astype(str).tolist()

def parse_sentiment(result: dict) -> tuple[int, float]:
    label = result['label'].upper()
    pred = 1 if 'POS' in label else 0
    return pred, result['score']

print(f"{len(texts)}건 감성 분류 진행 중... (배치 추론)")
results = en_classifier(texts, batch_size=16)

parsed = [parse_sentiment(r) for r in results]
df_sample['predicted'] = [p for p, _ in parsed]
df_sample['confidence'] = [c for _, c in parsed]

print("분류 완료!")
display(df_sample[['review_clean', 'voted_up', 'predicted', 'confidence']].head(10))

87건 감성 분류 진행 중... (배치 추론)
분류 완료!


,review_clean,voted_up,predicted,confidence
0,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",True,0,0.609226
1,"this game is like a zen-garden, I love it! \n\...",True,1,0.966662
2,Ever played Bhop? Surf? If so this games mecha...,True,1,0.959454
3,"Greatness comes in all sorts of things, but th...",True,1,0.951157
4,Don't buy this game. The platforms you jump on...,False,0,0.899224
5,Enjoying this game SO MUCH !,True,1,0.988802
6,I agree with the other reviews which praise th...,True,1,0.726997
7,Your Game Sucks I'm Refunding Bye,False,0,0.942208
8,"Gameplay can be difficult, but only enough tha...",True,1,0.952418
9,"A really stylish, pared-down bunnyhopping/park...",True,1,0.618744


In [ ]:
# ========================================
# 분류 성능 평가 리포트를 출력합니다
# ========================================

y_true = df_sample['voted_up'].astype(int).values # 실제 정답 라벨
y_pred = df_sample['predicted'].values   # 모델이 예측한 라벨

print("[분류 성능 평가 리포트]")
print("=" * 60)

print(classification_report(
    y_true, y_pred,
    target_names=['부정 (0)', '긍정 (1)'],  # 라벨 이름
    digits=4                              # 소수점 4자리까지 표시
))

# 주요 지표를 요약합니다
acc = accuracy_score(y_true, y_pred)           # 전체 정확도
f1 = f1_score(y_true, y_pred, average='weighted')  # 가중 F1-Score
print(f"전체 정확도 (Accuracy): {acc:.4f} ({acc*100:.1f}%)")
print(f"가중 F1-Score: {f1:.4f}")

[분류 성능 평가 리포트]
              precision    recall  f1-score   support

      부정 (0)     0.5143    0.9000    0.6545        20
      긍정 (1)     0.9615    0.7463    0.8403        67

    accuracy                         0.7816        87
   macro avg     0.7379    0.8231    0.7474        87
weighted avg     0.8587    0.7816    0.7976        87

전체 정확도 (Accuracy): 0.7816 (78.2%)
가중 F1-Score: 0.7976
